In [1]:
# ===== 1: 설정 =====
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

PROJECT_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot"
BASE_DIR = PROJECT_DIR
CHUNKS_DIR = BASE_DIR / "data" / "chunks"
GRAPH_DIR = BASE_DIR / "data" / "graph"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_FILES = {
    "강아지": CHUNKS_DIR / "petmint_강아지.json",
    "고양이": CHUNKS_DIR / "petmint_고양이.json",
}

def load_chunks(species: str) -> list[dict]:
    with open(CHUNK_FILES[species], "r", encoding="utf-8") as f:
        return json.load(f)

chunks_dog = load_chunks("강아지")
chunks_cat = load_chunks("고양이")

print("강아지 청크 수:", len(chunks_dog))
print("고양이 청크 수:", len(chunks_cat))

강아지 청크 수: 928
고양이 청크 수: 850


In [2]:
# ===== 2: 조 번호 → 제목/유형 사전 + article_block_id 관련 사전 =====

def build_article_title_dict(chunks: list[dict]) -> dict:
    titles = {}
    for c in chunks:
        if c.get("article_num") is None:
            continue
        key = (c["section"], c.get("section_name"), c["article_num"])
        if key not in titles:
            titles[key] = {"title": c.get("article_title"), "type": c.get("type")}
    return titles


def build_block_dicts(chunks: list[dict]):
    """article_block_id 기준 사전들.
    - combined: bid -> 그 조에 속한 content 전부 합친 문자열 (정의 박스 탐색용)
    - block_order: bid 등장 순서(문서 물리 순서)
    - block_index: bid -> 순서 인덱스 (O(1) 조회, F)
    - block_paras: bid -> 그 조에 실재하는 paragraph_num 집합 (self_paragraph 해석용, B)
    """
    block_contents = defaultdict(list)
    block_order = []
    block_paras = defaultdict(set)
    seen = set()
    for c in chunks:
        bid = c.get("article_block_id")
        if bid is None:
            continue
        block_contents[bid].append(c.get("content") or "")
        block_paras[bid].add(c.get("paragraph_num"))
        if bid not in seen:
            seen.add(bid)
            block_order.append(bid)
    combined = {bid: "\n".join(texts) for bid, texts in block_contents.items()}
    block_index = {bid: i for i, bid in enumerate(block_order)}
    return combined, block_order, block_index, block_paras


article_titles_dog = build_article_title_dict(chunks_dog)
article_titles_cat = build_article_title_dict(chunks_cat)

block_content_dog, block_order_dog, block_index_dog, block_paras_dog = build_block_dicts(chunks_dog)
block_content_cat, block_order_cat, block_index_cat, block_paras_cat = build_block_dicts(chunks_cat)

print("강아지 조 번호 사전 개수:", len(article_titles_dog))
print("고양이 조 번호 사전 개수:", len(article_titles_cat))
print("강아지 article_block_id 개수:", len(block_content_dog))
print("고양이 article_block_id 개수:", len(block_content_cat))


강아지 조 번호 사전 개수: 222
고양이 조 번호 사전 개수: 194
강아지 article_block_id 개수: 235
고양이 article_block_id 개수: 207


In [3]:
# ===== 3: citations 해석 도구 =====

LAW_SECTION_NAMES = {
    "강아지": {
        "개인정보 보호법": "【법규1】 개인정보 보호법",
        "금융소비자 보호에 관한 법률": "【법규2】 금융소비자 보호에 관한 법률",
        "민법": "【법규3】 민법",
        "신용정보의 이용 및 보호에 관한 법률": "【법규4】 신용정보의 이용 및 보호에 관한 법률",
        "약사법": "【법규5】 약사법",
        "민사소송법": "【법규6】 민사소송법",
        "소비자기본법": "【법규7】 소비자기본법",
    },
    "고양이": {
        "개인정보 보호법": "【법규1】 개인정보 보호법",
        "금융소비자 보호에 관한 법률": "【법규2】 금융소비자 보호에 관한 법률",
        "민법": "【법규3】 민법",
        "신용정보의 이용 및 보호에 관한 법률": "【법규4】 신용정보의 이용 및 보호에 관한 법률",
        "약사법": "【법규5】 약사법",
        "민사소송법": "【법규6】 민사소송법",
        "소비자기본법": "【법규7】 소비자기본법",
    },
}

NEARBY_RANGE = 10


def law_section_name_of(law_name: str, species: str):
    key = re.sub(r"\s+", "", law_name or "")
    return next((sname for name, sname in LAW_SECTION_NAMES.get(species, {}).items()
                 if re.sub(r"\s+", "", name) == key), None)


# 준용규정의 제외 목록: '다만, 보통약관 제3조(…), 제4조(…) … 및 제38조(…)은 제외합니다' — 이 조문들은 적용되지 않으므로
# 참조로 붙이면 적용되지 않는 조문을 적용되는 것처럼 설명하게 된다(2026-10-12 측정: 해당 FAQ 9개 전부 오류).
# '단, …에 따라 … 적용하지 않습니다'(조건 근거)나 법령 박스 안의 '제외한다'는 해당하지 않는다 — 종마다 3문장(일반조항 22조, 배상 17조, 민사 12조)
EXCLUSION_LIST = re.compile(r"다만,((?:보통약관)?(?:제1절일반조항)?제\d+조[^.]*?)(?:은|는)제외합니다")


def excluded_articles(content: str) -> set[int]:
    flat = re.sub(r"\s+", "", content or "")       # '보 통약관'처럼 줄바꿈으로 끊긴 낱말까지 잇는다
    nums = set()
    for m in EXCLUSION_LIST.finditer(flat):
        nums |= {int(n) for n in re.findall(r"제(\d+)조", m.group(1))}
    return nums


def resolve_named_document(cit, article_titles):
    """'보통약관 제N조', '「…일반조항」 제N조'처럼 문서를 명시한 인용(01 doc_name) — 그 문서에서만 찾는다.
    문서명을 보지 않고 같은 번호 조를 찾으면 '반려동물 비용손해 관련 특별약관 일반조항 제7조(계약 전 알릴 의무)'가
    배상책임 일반조항 제7조(보험금의 분담)로 연결되는 식의 틀린 연결이 생긴다(2026-10-12 측정 4/3건).
    반환 (scope, ref_section, ref_section_name, ref_article_num, title)."""
    doc = cit["doc_name"]
    sec, sname = ("보통약관", None) if doc == "보통약관" else ("특별약관", doc)
    info = article_titles.get((sec, sname, cit.get("article_num")))
    if info:
        return ("named_document", sec, sname, cit["article_num"], info["title"])
    return ("not_found", None, None, None, None)


def find_inline_law_box(law_name: str, article_num: int, content: str) -> bool:
    """content 안에 '【법명 제N조...】' 형태의 정의 박스가 있는지 확인"""
    pattern = re.compile(rf"【\s*[「『]?\s*{re.escape(law_name)}\s*제\s*{article_num}\s*조")  # 【 뒤 낫표(「『) 허용
    return bool(pattern.search(content))


def resolve_law_citation(cit, c, block_content, block_order, block_index, block_paras, species, article_titles):
    """law_name citation 해석. 반환 (scope, ref_section, ref_section_name, ref_article_num).
    - 조번호 없는 항참조는 같은 조(block) 안의 항을 가리키는 self_paragraph.
    - inline 정의 박스(같은 조/근접 조) → inline_law_* (그래프 제외).
    - 법규 섹션에 해당 법 조문이 수록돼 있으면 law_section_reference.
    - 어디서도 못 찾으면(외부법 미수록 등) law_not_in_excerpt (= 미수록, 'not_found'=조회실패와 구분).
    """
    law_name = cit["law_name"]
    article_num = cit.get("article_num")
    para = cit.get("paragraph_num")
    bid = c.get("article_block_id")

    # 조번호 없는 호 → 지금 항의 열거 항목 (self_item, 그래프 제외)
    if article_num is None and cit.get("paragraph_type") == "호":
        return ("self_item", None, None, None)
    # 조번호 없는 항참조 → 같은 조 안의 항 (self)
    if article_num is None:
        if para is not None and bid is not None and para in block_paras.get(bid, set()):
            return ("self_paragraph", c["section"], c.get("section_name"), c.get("article_num"))
        return ("law_not_in_excerpt", None, None, None)

    # inline 정의 박스 (같은 조)
    if bid is not None and bid in block_content and find_inline_law_box(law_name, article_num, block_content[bid]):
        return ("inline_law_same_block", None, None, None)
    # inline 정의 박스 (근접 조)
    if bid is not None and bid in block_index:
        idx = block_index[bid]
        nearby = block_order[max(0, idx - NEARBY_RANGE):idx] + block_order[idx + 1:idx + 1 + NEARBY_RANGE]
        for nid in nearby:
            if find_inline_law_box(law_name, article_num, block_content.get(nid, "")):
                return ("inline_law_nearby", None, None, None)

    # 법규 섹션에 수록돼 있나 (E: 넘겨받은 species 사전만 조회). 법명은 공백을 무시하고 비교한다
    # ('금융소비자보호에 관한법률' = '금융소비자 보호에 관한 법률' — PDF 줄바꿈·띄어쓰기 차이)
    law_section_name = law_section_name_of(law_name, species)
    if law_section_name and ("법규", law_section_name, article_num) in article_titles:
        return ("law_section_reference", "법규", law_section_name, article_num)

    return ("law_not_in_excerpt", None, None, None)


In [4]:
# ===== 4: citations 전체 해석 =====

def resolve_citations(chunks, article_titles, block_content, block_order, block_index, block_paras, species):
    for c in chunks:
        if not c.get("citations"):
            continue
        c_type = c.get("type")
        excluded = excluded_articles(c.get("content"))
        valid = []
        for cit in c["citations"]:
            cited_num = cit.get("article_num")

            # ===== 준용규정 제외 목록의 조문 → 적용되지 않으므로 참조하지 않는다(그래프 제외) =====
            if cit.get("doc_name") == "보통약관" and cited_num in excluded:
                cit["matched_scope"] = "excluded_reference"
                cit["ref_section"] = cit["ref_section_name"] = cit["ref_article_num"] = None
                valid.append(cit)
                continue

            # ===== law_name citation =====
            if cit.get("law_name"):
                scope, rsec, rsname, rnum = resolve_law_citation(
                    cit, c, block_content, block_order, block_index, block_paras, species, article_titles)
                cit["matched_scope"] = scope
                cit["ref_section"] = rsec
                cit["ref_section_name"] = rsname
                cit["ref_article_num"] = rnum
                if scope == "self_paragraph":
                    cit["article_title"] = c.get("article_title")
                valid.append(cit)
                continue

            # ===== 다른 약관 문서를 명시한 인용 → 그 문서에서만 =====
            if cit.get("doc_name"):
                scope, rsec, rsname, rnum, rtitle = resolve_named_document(cit, article_titles)
                if scope == "named_document" and (rsec, rsname, rnum) == (c["section"], c.get("section_name"), c.get("article_num")):
                    continue  # 자기 조
                cit["matched_scope"] = scope
                cit["ref_section"], cit["ref_section_name"], cit["ref_article_num"] = rsec, rsname, rnum
                if rtitle:
                    cit["article_title"] = rtitle
                valid.append(cit)
                continue

            # ===== 단독 호(조·항 없음) → 지금 항의 열거 항목. 자기 본문이라 참조하지 않는다(그래프 제외) =====
            # 호 번호를 항으로 조회하면 내용이 전혀 다른 항이 붙는다(2026-10-12 측정 35/33건)
            if cited_num is None and cit.get("paragraph_type") == "호":
                cit["matched_scope"] = "self_item"
                cit["ref_section"] = cit["ref_section_name"] = cit["ref_article_num"] = None
                valid.append(cit)
                continue

            # ===== 단독 항/호(조번호 없음) → 같은 조(block) 안의 항 (B) =====
            if cited_num is None:
                para = cit.get("paragraph_num")
                bid = c.get("article_block_id")
                if para is not None and bid is not None and para in block_paras.get(bid, set()):
                    cit["matched_scope"] = "self_paragraph"
                    cit["ref_section"] = c["section"]
                    cit["ref_section_name"] = c.get("section_name")
                    cit["ref_article_num"] = c.get("article_num")
                    cit["article_title"] = c.get("article_title")
                else:
                    cit["matched_scope"] = "self_paragraph_not_found"
                    cit["ref_section"] = None
                    cit["ref_section_name"] = None
                    cit["ref_article_num"] = None
                valid.append(cit)
                continue

            if cited_num == c.get("article_num"):
                continue  # 자기 조

            # ===== 같은 section, 같은 section_name =====
            key_same = (c["section"], c.get("section_name"), cited_num)
            if key_same in article_titles:
                cit["article_title"] = article_titles[key_same]["title"]
                cit["matched_scope"] = "same_section"
                cit["ref_section"] = c["section"]
                cit["ref_section_name"] = c.get("section_name")
                cit["ref_article_num"] = cited_num
                valid.append(cit)
                continue

            # ===== 같은 section, 다른 section_name: type 일치 우선 =====
            same_section_candidates = [
                (sec, sname, info) for (sec, sname, num), info in article_titles.items()
                if sec == c["section"] and num == cited_num
            ]
            if same_section_candidates:
                m = next((cand for cand in same_section_candidates if cand[2]["type"] == c_type), None)
                chosen = m if m else same_section_candidates[0]
                cit["article_title"] = chosen[2]["title"]
                cit["matched_scope"] = "same_section_same_type" if m else "same_section_diff_type"
                cit["ref_section"] = chosen[0]
                cit["ref_section_name"] = chosen[1]
                cit["ref_article_num"] = cited_num
                valid.append(cit)
                continue

            # ===== 완전히 다른 section (보통약관 우선) =====
            candidates = [(sec, sname, info) for (sec, sname, num), info in article_titles.items() if num == cited_num]
            if candidates:
                priority = next((cand for cand in candidates if cand[0] == "보통약관"), candidates[0])
                cit["article_title"] = priority[2]["title"]
                cit["matched_scope"] = "cross_section"
                cit["ref_section"] = priority[0]
                cit["ref_section_name"] = priority[1]
                cit["ref_article_num"] = cited_num
                valid.append(cit)
                continue

            cit["matched_scope"] = "not_found"
            cit["ref_section"] = None
            cit["ref_section_name"] = None
            cit["ref_article_num"] = None
            valid.append(cit)

        c["citations"] = valid
    return chunks


chunks_dog = resolve_citations(chunks_dog, article_titles_dog, block_content_dog, block_order_dog, block_index_dog, block_paras_dog, "강아지")
chunks_cat = resolve_citations(chunks_cat, article_titles_cat, block_content_cat, block_order_cat, block_index_cat, block_paras_cat, "고양이")

scope_dist = Counter(cit.get("matched_scope") for c in chunks_dog for cit in c.get("citations", []))
print(scope_dist)


Counter({'self_paragraph': 498, 'same_section': 105, 'law_not_in_excerpt': 101, 'named_document': 99, 'law_section_reference': 49, 'inline_law_same_block': 47, 'self_item': 39, 'excluded_reference': 17, 'inline_law_nearby': 6, 'cross_section': 5})


In [5]:
count_total = 0
count_with_real_circles = 0
for c in chunks_dog:
    if c.get("section") == "법규":
        count_total += 1
        content = c.get("content") or ""
        if any(ch in content for ch in "①②③④⑤⑥⑦⑧⑨⑩"):
            count_with_real_circles += 1

print(f"법규 섹션 전체 청크: {count_total}건")
print(f"그중 진짜 원문자(①②③...)가 섞여있는 청크: {count_with_real_circles}건")

법규 섹션 전체 청크: 103건
그중 진짜 원문자(①②③...)가 섞여있는 청크: 91건


In [6]:
# ===== 4: 결과 저장 (citations가 보강된 청크 전체) =====

with open(GRAPH_DIR / "petmint_강아지_resolved.json", "w", encoding="utf-8") as f:
    json.dump(chunks_dog, f, ensure_ascii=False, indent=2)

with open(GRAPH_DIR / "petmint_고양이_resolved.json", "w", encoding="utf-8") as f:
    json.dump(chunks_cat, f, ensure_ascii=False, indent=2)

print("저장 완료:", GRAPH_DIR)

저장 완료: /home/kopo17/AI_programming/petmint-rag-chatbot/data/graph
